# LIBS Instrument Calibration Report

**Instrument:** <INSTRUMENT_NAME>

This report summarizes the instrument profile calibration process. It compares the measured experimental spectra with a synthetic spectrum generated using a two-zone plasma model (Hot Core + Cool Periphery).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# ==========================================
# CONFIGURATION PARAMETERS (Injected by Java)
# ==========================================

INSTRUMENT_NAME = "<INSTRUMENT_NAME>"
INPUT_CSV_PATH = r"<INPUT_CSV_PATH>"
PROCESSED_CSV_PATH = r"<PROCESSED_SPECTRUM_PATH>"
HOT_SPECTRUM_PATH = r"<HOT_SPECTRUM_PATH>"
COOL_SPECTRUM_PATH = r"<COOL_SPECTRUM_PATH>"

# Optimized Plasma Parameters
HOT_TE = <HOT_CORE_TE>
HOT_NE = <HOT_CORE_NE>
COOL_TE = <COOL_PERIPHERY_TE>
COOL_NE = <COOL_PERIPHERY_NE>
WEIGHT = <WEIGHT>  # Weight for Hot Core (w). Total = w*Hot + (1-w)*Cool

print(f"Calibration Report for: {INSTRUMENT_NAME}")
print(f"Optimized Parameters:")
print(f"  Hot Core: Te={HOT_TE} eV, Ne={HOT_NE:e} cm^-3")
print(f"  Cool Periphery: Te={COOL_TE} eV, Ne={COOL_NE:e} cm^-3")
print(f"  Weight (Hot Core): {WEIGHT:.4f}")

## 1. Experimental Data Visualization
### 1.1 Raw Measured Spectra (All Shots)

In [ ]:
if os.path.exists(INPUT_CSV_PATH):
    try:
        # Attempt to read CSV. Assuming standard format with headers.
        # If the first row is wavelengths, we might need header=None or specific handling.
        # Based on typical LIBS data, rows are shots, columns are wavelengths.
        raw_df = pd.read_csv(INPUT_CSV_PATH)
        
        # Check if columns are numeric (wavelengths)
        try:
            wavelengths = raw_df.columns.astype(float)
        except ValueError:
            # If headers are not wavelengths, maybe it's transposed or has metadata
            # Assuming standard case for now as per spec "columns represent wavelength channels"
            wavelengths = np.arange(len(raw_df.columns)) # Fallback

        plt.figure(figsize=(12, 6))
        plt.title(f"Raw Measured Spectra - {INSTRUMENT_NAME}")
        plt.xlabel("Wavelength (nm)")
        plt.ylabel("Intensity (a.u.)")

        # Plot all shots (limit to first 50 if too many to avoid clutter)
        for index, row in raw_df.iterrows():
            if index > 50: break
            plt.plot(wavelengths, row, alpha=0.3, linewidth=0.5, color='gray')

        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Error loading raw data: {e}")
else:
    print(f"Input file not found: {INPUT_CSV_PATH}")

### 1.2 Preprocessed Spectrum (Averaged & Baseline Corrected)
This spectrum serves as the target for the calibration optimization.

In [ ]:
target_wavelengths = None
target_intensity = None

if os.path.exists(PROCESSED_CSV_PATH):
    try:
        processed_df = pd.read_csv(PROCESSED_CSV_PATH)
        
        # Identify Wavelength and Intensity columns
        # Expecting either columns named 'Wavelength', 'Intensity' or just two columns
        cols = processed_df.columns
        if 'wavelength' in [c.lower() for c in cols] and 'intensity' in [c.lower() for c in cols]:
             # Find exact col names
             w_col = next(c for c in cols if c.lower() == 'wavelength')
             i_col = next(c for c in cols if c.lower() == 'intensity')
             target_wavelengths = processed_df[w_col].values
             target_intensity = processed_df[i_col].values
        elif len(cols) >= 2:
             target_wavelengths = processed_df.iloc[:, 0].values
             target_intensity = processed_df.iloc[:, 1].values
        
        if target_wavelengths is not None:
            plt.figure(figsize=(12, 6))
            plt.title("Preprocessed Target Spectrum")
            plt.xlabel("Wavelength (nm)")
            plt.ylabel("Intensity (a.u.)")
            plt.plot(target_wavelengths, target_intensity, color='black', label='Target Spectrum')
            plt.legend()
            plt.tight_layout()
            plt.show()
    except Exception as e:
        print(f"Error loading processed data: {e}")
else:
    print(f"Processed file not found: {PROCESSED_CSV_PATH}")

## 2. Synthetic Zone Spectra
Individual spectra generated for the optimized Hot Core and Cool Periphery parameters.

In [ ]:
hot_df = None
cool_df = None

def load_spectrum(path):
    if os.path.exists(path):
        return pd.read_csv(path)
    return None

hot_df = load_spectrum(HOT_SPECTRUM_PATH)
cool_df = load_spectrum(COOL_SPECTRUM_PATH)

plt.figure(figsize=(12, 6))
plt.title("Synthetic Zone Spectra")
plt.xlabel("Wavelength (nm)")
plt.ylabel("Intensity (a.u.)")

if hot_df is not None and len(hot_df.columns) >= 2:
     plt.plot(hot_df.iloc[:,0], hot_df.iloc[:,1], color='red', alpha=0.7, label=f'Hot Core (Te={HOT_TE})')

if cool_df is not None and len(cool_df.columns) >= 2:
     plt.plot(cool_df.iloc[:,0], cool_df.iloc[:,1], color='blue', alpha=0.7, label=f'Cool Periphery (Te={COOL_TE})')

plt.legend()
plt.tight_layout()
plt.show()

## 3. Calibration Result & Validation
Overlay of the measured spectrum with the final weighted synthetic spectrum.

**Formula:** $I_{total} = w \cdot I_{hot} + (1-w) \cdot I_{cool}$

In [ ]:
if hot_df is not None and cool_df is not None and target_wavelengths is not None:
    
    # Interpolate synthetic spectra to match target wavelength grid
    hot_interp = np.interp(target_wavelengths, hot_df.iloc[:,0], hot_df.iloc[:,1])
    cool_interp = np.interp(target_wavelengths, cool_df.iloc[:,0], cool_df.iloc[:,1])
    
    # Calculate Combined Spectrum
    combined_intensity = (WEIGHT * hot_interp) + ((1 - WEIGHT) * cool_interp)
    
    # --- PLOT ---
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10), sharex=True, gridspec_kw={'height_ratios': [3, 1]})
    
    # Main Comparison
    ax1.set_title(f"Calibration Fit (w={WEIGHT:.3f}) - {INSTRUMENT_NAME}")
    ax1.plot(target_wavelengths, target_intensity, color='black', alpha=0.8, label='Measured (Processed)')
    ax1.plot(target_wavelengths, combined_intensity, color='green', linestyle='--', linewidth=1.5, label='Synthetic Fit')
    ax1.set_ylabel("Intensity (a.u.)")
    ax1.legend(loc='upper right')
    ax1.grid(True, alpha=0.3)
    
    # Residuals
    residuals = target_intensity - combined_intensity
    ax2.set_title("Residuals (Measured - Synthetic)")
    ax2.plot(target_wavelengths, residuals, color='purple', linewidth=1)
    ax2.axhline(0, color='gray', linestyle='-', linewidth=0.5)
    ax2.set_xlabel("Wavelength (nm)")
    ax2.set_ylabel("Difference")
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # --- STATISTICS ---
    mse = np.mean(residuals**2)
    rmse = np.sqrt(mse)
    # Calculate R-squared
    ss_res = np.sum(residuals**2)
    ss_tot = np.sum((target_intensity - np.mean(target_intensity))**2)
    r2 = 1 - (ss_res / ss_tot)
    
    print(f"Fit Statistics:")
    print(f"  MSE:  {mse:.6f}")
    print(f"  RMSE: {rmse:.6f}")
    print(f"  R^2:  {r2:.6f}")

else:
    print("Cannot generate validation plot. Missing necessary data streams.")